# Tema 6 · La primera GPU
*Curso «Google Colab» · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** mover datos a la GPU, hacer cálculos ahí y **medir bien** cuánto más rápida es.

> Abre este cuaderno con GPU (**Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**). Si no hay GPU
> disponible en el plan gratis, el cuaderno corre igual en CPU.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
# La prueba automática de MiAtlas.cl define MIATLAS_PRUEBA=1 para correr con pocos datos. Tú no necesitas tocarla.
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO, "| prueba rápida:", PRUEBA_RAPIDA)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU: el cuaderno funciona, pero más lento. En Colab: Entorno de ejecución → Cambiar tipo → GPU.")

## 1. Tensores: los arreglos de PyTorch

Un **tensor** es como un arreglo de NumPy, pero puede vivir en la **CPU** o en la **GPU**. `.to(DISPOSITIVO)` lo mueve.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(a, a.device)
a_gpu = a.to(DISPOSITIVO)
print(a_gpu.device)
print("a @ a =", (a_gpu @ a_gpu).cpu())      # .cpu() lo trae de vuelta para mostrarlo

**Regla de oro:** dos tensores que se operan juntos deben estar **en el mismo dispositivo**. Si uno está en la GPU y
el otro en la CPU, PyTorch da un error (`Expected all tensors to be on the same device`).

## 2. Medir bien: CPU contra GPU

La GPU trabaja **en paralelo y de forma asíncrona**: PyTorch le entrega la tarea y sigue. Para medir el tiempo real
hay que esperar a que termine con `torch.cuda.synchronize()`. La primera llamada además «calienta» la GPU, así que no
se mide.

In [ ]:
import time

def medir(n, dispositivo, repeticiones=3):
    x = torch.randn(n, n, device=dispositivo)
    y = torch.randn(n, n, device=dispositivo)
    x @ y                                         # calentamiento: no se mide
    if dispositivo == "cuda":
        torch.cuda.synchronize()
    inicio = time.perf_counter()
    for _ in range(repeticiones):
        x @ y
    if dispositivo == "cuda":
        torch.cuda.synchronize()                  # esperar a que la GPU termine de verdad
    return (time.perf_counter() - inicio) / repeticiones

N = 512 if PRUEBA_RAPIDA else 4096
t_cpu = medir(N, "cpu")
print(f"Multiplicar dos matrices de {N}×{N} en CPU: {t_cpu * 1000:.1f} ms")
if DISPOSITIVO == "cuda":
    t_gpu = medir(N, "cuda")
    print(f"En GPU: {t_gpu * 1000:.1f} ms  →  unas {t_cpu / t_gpu:.0f} veces más rápido")
else:
    print("Sin GPU no hay comparación: vuelve a probar cuando Colab te asigne una.")

> **Experimenta:** cambia `N` a 256 y a 8192. Con matrices chicas la GPU casi no gana: mover los datos y lanzar la
> tarea cuesta más que el cálculo. La GPU brilla con **mucho trabajo en paralelo**, como entrenar redes neuronales.

## 3. Cuánta memoria usa la GPU

In [ ]:
if DISPOSITIVO == "cuda":
    libre, total = torch.cuda.mem_get_info()
    print(f"Memoria de la GPU: {(total - libre) / 1024**3:.2f} GB usados de {total / 1024**3:.1f} GB")
    grande = torch.randn(8192, 8192, device="cuda")    # unos 256 MB
    print(f"Con un tensor grande: {torch.cuda.memory_allocated() / 1024**2:.0f} MB reservados por PyTorch")
    del grande
    torch.cuda.empty_cache()                           # devolver la memoria
else:
    print("Esta sección necesita GPU.")

## Pruébalo tú

Escribe `en_gpu(x)`, que reciba un tensor y devuelva **una copia en `DISPOSITIVO`** elevada al cuadrado (`x ** 2`).

In [ ]:
def en_gpu(x):
    # ← tu código aquí
    return None

In [ ]:
# Comprobación
r = en_gpu(torch.tensor([1.0, 2.0, 3.0]))
if r is None:
    print("Aún no lo resuelves.")
elif r.device.type == DISPOSITIVO and torch.equal(r.cpu(), torch.tensor([1.0, 4.0, 9.0])):
    print("✅ ¡Correcto!, y quedó en", r.device)
else:
    print("❌ Revisa el dispositivo o el cálculo:", r)

## Resumen

- `.to(DISPOSITIVO)` mueve tensores (y modelos) a la GPU; todo lo que se opera junto va en el mismo dispositivo.
- Para medir la GPU: calentar primero y `torch.cuda.synchronize()` antes de parar el reloj.
- La GPU gana con mucho trabajo en paralelo; con poco, ni se nota.

**Fuentes:** [PyTorch · tensores](https://docs.pytorch.org/docs/stable/tensors.html) · [PyTorch · semántica CUDA y tiempos asíncronos](https://docs.pytorch.org/docs/stable/notes/cuda.html)